In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 8.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [3]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [4]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://reawake-brilliant-favorable.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://reawake-brilliant-favorable.ngrok-free.dev


True

In [5]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [6]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [7]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。學校的校名「明新」取自《大學》「在明明德，在新民，在止於至善」的精義，旨在闡揚人類與生俱來的德性與情操，並期望學子能涵養高尚品德、擁有專業學問與優良技術，進而達到全人發展的境界。

**歷史沿革**
明新科技大學的創立可追溯至1966年，當時以「明新工業專科學校」之名成立。 隨著教育體制的發展，學校於1997年改制為「明新技術學院」並附設專科部。 最終在2002年9月，奉教育部核准升格為「明新科技大學」，並於2018年12月更名為「明新學校財團法人明新科技大學」，展現其全新的發展格局。

**校訓與教育理念**
明新科技大學秉持「堅毅、求新、創造」的校訓。 學校以「多元學習」、「全球視野」、「永續經營」與「技術創新」為四大辦學理念，致力於培育具備「跨域整合、務實創新、全人學習」的專業人才，並以成為「國際魅力產業科技大學」為發展願景。

**學術組織**
目前，明新科技大學設有六個學院，包括：
*   半導體學院
*   工程學院
*   管理學院
*   民生學院
*   人文與設計學院
*   共同教育學院

學校涵蓋約20個學系、2個學位學程（包含1個博士學位學程）及11個碩士班。 值得一提的是，學校於2022年10月獲教育部核准通過「半導體科技博士學位學程」，這是該校成立56年來的第一個博士班。

**地理位置與特色**
明新科技大學佔地逾三十公頃，地處新竹縣新豐鄉，依傍省道縱貫線，毗鄰中山高速公路，交通便利。 學校鄰近新竹工業區與科學園區，憑藉地理優勢，多年來與產業界建立了多項產學合作計畫，為區域產業培育了大量優秀人才，被視為新竹縣人才供應的重要學府。 校園環境清幽，四周山青樹碧，日間可遠眺臺灣海峽，夜間可俯瞰竹塹夜景，為學子提供良好的學習環境。


In [8]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長為呂明峯。他於2025年1月16日舉行布達暨交接典禮，並於2月1日正式上任，成為該校第11任校長。

呂明峯校長在明新科技大學服務已有34年，擁有豐富的業界經驗和深厚的學術背景。 他在任內曾打造全台首座半導體封裝測試類產線，並推動成立半導體學院。 他提出了學校未來四大發展方向，旨在將明新科大打造成新竹地方人才庫、桃竹苗大矽谷的推動引擎、新南向專班的基地，並活化資源以實現永續校園。


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616524811321737576","quoteToken":"EBrIx2FpNx8fW-6BGN5FeQHzLd6XZzSVwndCKEkLoTd1xW_uS70Wr0QzsglKbmYfR3yoEFvAZVYBt5ny9xm54-lIGwglUzCaK2Rp1sZsHwMSfcjc4bKahPXOL-Tefpw7Ljxq03XEaC4DXaC_jJEafw","markAsReadToken":"4J_zPJNNLkl8Wg5zpemGny0Mo0q1jwHkHXQO4qCK5TF6TnzkF3WGb10sASLrlFPNwNIogqxJE4F20m-IF9R4txlKZUwtotLp-1BT7PgTVby2fHVjhYX7Doj-NE7WL5_WZyI2vTD68AQ7y8OtmLw1MhaT5zdQJqHw5ZlvOByc6M--MiLWFEENqa5qOdFjaDY7IZxcJk9QDu3ZnI84GxvQOA","text":"AI 校長愛吃甚麼？"},"webhookEventId":"01KT1APPCMN6ZB73Y54JQ4EAD2","deliveryContext":{"isRedelivery":false},"timestamp":1780308727778,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"7e6b03c5056e48fe835

BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616524811321737576","quoteToken":"EBrIx2FpNx8fW-6BGN5FeQHzLd6XZzSVwndCKEkLoTd1xW_uS70Wr0QzsglKbmYfR3yoEFvAZVYBt5ny9xm54-lIGwglUzCaK2Rp1sZsHwMSfcjc4bKahPXOL-Tefpw7Ljxq03XEaC4DXaC_jJEafw","markAsReadToken":"4J_zPJNNLkl8Wg5zpemGny0Mo0q1jwHkHXQO4qCK5TF6TnzkF3WGb10sASLrlFPNwNIogqxJE4F20m-IF9R4txlKZUwtotLp-1BT7PgTVby2fHVjhYX7Doj-NE7WL5_WZyI2vTD68AQ7y8OtmLw1MhaT5zdQJqHw5ZlvOByc6M--MiLWFEENqa5qOdFjaDY7IZxcJk9QDu3ZnI84GxvQOA","text":"AI 校長愛吃甚麼？"},"webhookEventId":"01KT1APPCMN6ZB73Y54JQ4EAD2","deliveryContext":{"isRedelivery":false},"timestamp":1780308727778,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"7e6b03c5056e48fe83511fccef2dd15d","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 10:12:11] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616524928392888951","quoteToken":"5evjDkPXWi6nSFNX3PVIG7SL0CmWCWB-6BbV9lMRH2uowGFp2h0NTtwjRmEy8CKuDU92bDNaCftly_4aMJ5FWEeHUGM6SPar-JBetGvt1jSzhrXtCY286CWS1X_aiLLiszttA0ApZMEkOwFqYYXlHw","markAsReadToken":"R8_s_YhtLbxWWsGPdo83Btj2PnTc0712qwWqREQF3BkfFCPF6CMwqEKVD7559KeCUz7RASPFwbKDApFXPPDhPN9t_72mAqN79zLU-hPIC_JjszuLPRG8tJllHNvR8rTmFsB3BkbfkJ_e065t_PBRUhXhBvmSETNTvhrZ5sGyF-Q_-AWV9pljUJ_TnaFgxB847utUbU57GerzTehvHG70qg","text":"AI 校長愛吃甚麼？"},"webhookEventId":"01KT1ARTPNQZEP43HYYC5MZEF5","deliveryContext":{"isRedelivery":false},"timestamp":1780308797658,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"cb9f63faeaca44a298aa29d371ce0c23","mode":"active"}]}


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616524928392888951","quoteToken":"5evjDkPXWi6nSFNX3PVIG7SL0CmWCWB-6BbV9lMRH2uowGFp2h0NTtwjRmEy8CKuDU92bDNaCftly_4aMJ5FWEeHUGM6SPar-JBetGvt1jSzhrXtCY286CWS1X_aiLLiszttA0ApZMEkOwFqYYXlHw","markAsReadToken":"R8_s_YhtLbxWWsGPdo83Btj2PnTc0712qwWqREQF3BkfFCPF6CMwqEKVD7559KeCUz7RASPFwbKDApFXPPDhPN9t_72mAqN79zLU-hPIC_JjszuLPRG8tJllHNvR8rTmFsB3BkbfkJ_e065t_PBRUhXhBvmSETNTvhrZ5sGyF-Q_-AWV9pljUJ_TnaFgxB847utUbU57GerzTehvHG70qg","text":"AI 校長愛吃甚麼？"},"webhookEventId":"01KT1ARTPNQZEP43HYYC5MZEF5","deliveryContext":{"isRedelivery":false},"timestamp":1780308797658,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"cb9f63faeaca44a298aa29d371ce0c23","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 10:13:25] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"file","id":"616524980922089507","markAsReadToken":"uzvYUlSU16Th5xN5SQAdKckDrY9XpW49TCz_fF3YxQNOw6yjtXDgnr8usQ3D2Tj43fDD8HGPDOrAfuSMsVGfU-SdlIAwU7ugRsd0iAsvfjKu39_qA0loCg86zthSGwLikDDFbNElq_L1KcWIhYT45mPFHPvzGK2hWciy7MEmYMjFsEiKyEnozXuFJqPXM0qijEm1_rSadTeLwh9qExnLSQ","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KT1ASSK796HYA5ZAGHC0E7JR","deliveryContext":{"isRedelivery":false},"timestamp":1780308829742,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"373d7040e76447efba9386a89f293592","mode":"active"}]}


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"file","id":"616524980922089507","markAsReadToken":"uzvYUlSU16Th5xN5SQAdKckDrY9XpW49TCz_fF3YxQNOw6yjtXDgnr8usQ3D2Tj43fDD8HGPDOrAfuSMsVGfU-SdlIAwU7ugRsd0iAsvfjKu39_qA0loCg86zthSGwLikDDFbNElq_L1KcWIhYT45mPFHPvzGK2hWciy7MEmYMjFsEiKyEnozXuFJqPXM0qijEm1_rSadTeLwh9qExnLSQ","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KT1ASSK796HYA5ZAGHC0E7JR","deliveryContext":{"isRedelivery":false},"timestamp":1780308829742,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"373d7040e76447efba9386a89f293592","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/oy215xkgf4a0


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 10:13:53] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616525023485886677","quoteToken":"hvZUaJfKWFOnLBHYKu5ZEXR7ZwvgwWo1ha820oUJpYIg1PZ1lyFtb2xWtPNmNBt-X5i6j0fm0Loij2wsZKkaui86D9jypBYfnADfMbuL4Q-WiT5dJT2u3WMNowFWky3z8g8vzK59R8ovL9sOomR10w","markAsReadToken":"Bff3y3L5PxRHh_vXc0FMCWaI4WfUgeAyFu3ftgzpWydTVhWqUimmuzNkooylUA1NZQcJylYvuZMdcltzD2Ejsk3c7ANg2lWl-V45xTgMBUreKYGgYRa54d3odXcknjVCJDcGwarcqceXfQxHkKU1S_s9-JGP6XM6sW0BFW09ICwmGs7lutjWgN4PWT0-Z1p6h3qWNO0yIzXAKgdkMyjJww","text":"AI 校長愛吃甚麼？"},"webhookEventId":"01KT1ATHY1BKX6P30B3ZVRFZ6H","deliveryContext":{"isRedelivery":false},"timestamp":1780308854212,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"3ccd08c22a6546a1a880a45d56f33d7f","mode":"active"}]}


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616525023485886677","quoteToken":"hvZUaJfKWFOnLBHYKu5ZEXR7ZwvgwWo1ha820oUJpYIg1PZ1lyFtb2xWtPNmNBt-X5i6j0fm0Loij2wsZKkaui86D9jypBYfnADfMbuL4Q-WiT5dJT2u3WMNowFWky3z8g8vzK59R8ovL9sOomR10w","markAsReadToken":"Bff3y3L5PxRHh_vXc0FMCWaI4WfUgeAyFu3ftgzpWydTVhWqUimmuzNkooylUA1NZQcJylYvuZMdcltzD2Ejsk3c7ANg2lWl-V45xTgMBUreKYGgYRa54d3odXcknjVCJDcGwarcqceXfQxHkKU1S_s9-JGP6XM6sW0BFW09ICwmGs7lutjWgN4PWT0-Z1p6h3qWNO0yIzXAKgdkMyjJww","text":"AI 校長愛吃甚麼？"},"webhookEventId":"01KT1ATHY1BKX6P30B3ZVRFZ6H","deliveryContext":{"isRedelivery":false},"timestamp":1780308854212,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"3ccd08c22a6546a1a880a45d56f33d7f","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [01/Jun/2026 10:14:16] "POST / HTTP/1.1" 200 -
